# 압축 강도 증가에 따른 FPIR 급증 진단

목적: FIQA 성능 평가와 분리하여, 기존 완료 run에서 **어떤 검색 방식·임계값 정책에서 오수락이 증가하는지** 재현하고 사건·점수 변화로 분해합니다. 1차 구현은 기존 artifact 읽기 전용이며 FR 추론·압축기 재학습·threshold 선택을 수행하지 않습니다.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "research").is_dir())
SOURCE_MODEL = "arcface"  # arcface / adaface / magface / edgeface
PROFILES = tuple(f"pq_512_m{m}_b8" for m in (128, 64, 32, 16, 8))
MODES = ("pq_reconstruction_cosine", "pq_one_sided_cosine", "pq_adc_exhaustive")
TARGET_FPIRS = (0.01, 0.05, 0.10, 0.20, 0.30)
FOCUS_FPIR = 0.01
SEED = 8972  # 이번 진단 CI에만 사용; 과거 실험 seed는 변경하지 않음
BOOTSTRAP_RESAMPLES = 2000
WRITE_RESULTS = False
OUTPUT_ROOT = PROJECT_ROOT / "results" / "diagnostics" / "compression_fpir"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. 비교와 해석 규칙

- FPIR: 미등록 query의 gallery 최대 점수가 임계값 이상인 비율입니다. 개별 pair FMR이 아닙니다.
- H1: 기존 승자의 점수가 올라가 임계값을 넘는가? H2: 새 후보가 최대가 되면서 추가 상승하는가?
- H3: 원본 임계값을 유지할 때와 조건별 재보정할 때 결과가 다른가? H4: 양쪽 복원과 gallery-only 압축의 양상이 다른가?
- 모든 인접 압축 단계를 비교합니다. test로 유리한 threshold/seed를 선택하지 않습니다.
- ADC는 별도 점수 공간이므로 cosine과 점수를 빼거나 원본 cosine 임계값을 적용하지 않습니다.


In [ ]:
from IPython.display import display
from research.experiments.compression_fpir_failure_diagnosis import (
    SOURCE_RUNS, run_diagnosis, write_diagnosis, LIMITATIONS,
)
print("Pinned source:", PROJECT_ROOT / "runs" / SOURCE_RUNS[SOURCE_MODEL])
print("CPU artifact analysis only; no GPU inference.")
print(LIMITATIONS)


## 2. 출처 검증 및 재현

완료 run, v6 요약, 원장 파일 크기·SHA-256, 모델·프로토콜·query 집합을 검증합니다. 원장 TPIR20은 정답 점수 통과 AND Top-20인지 확인하고 요약 건수와 대조합니다. 파일이 없거나 불일치하면 다른 run으로 자동 대체하지 않고 중단합니다. 전체 5개 프로파일·3개 검색 방식의 기존 원장을 읽습니다.


In [ ]:
result = run_diagnosis(
    PROJECT_ROOT, model=SOURCE_MODEL, profiles=PROFILES, modes=MODES,
    targets=TARGET_FPIRS, seed=SEED, resamples=BOOTSTRAP_RESAMPLES,
)
summary = result["summary"]
focus = summary.loc[summary.target_fpir == FOCUS_FPIR].copy()
display(focus[["compression_profile", "search_mode", "threshold_policy",
               "n", "reference_fa", "candidate_fa", "origin_fpir", "compressed_fpir",
               "compressed_fpir_ci_low", "compressed_fpir_ci_high", "target_met_on_test",
               "compressed_tpir20_count", "mated_count", "compressed_tpir20",
               "origin_rank20", "compressed_rank20"]])


## 3. 신규·소멸 오수락과 인접 압축 단계

`new_fa`: 원본에서는 거절, 압축 후 오수락. `lost_fa`: 반대 사건. **압축 FA = 원본 FA + 신규 − 소멸**입니다. `both_fa`, `neither_fa`를 합하면 전체 미등록 query 수와 같습니다. 아래 CI는 고정 조건의 query 단위 탐색적 paired bootstrap입니다. identity 상관·calibration 분할·codec 변동·다중 비교는 포함하지 않습니다.


In [ ]:
display(focus[["compression_profile", "search_mode", "threshold_policy",
               "both_fa", "neither_fa", "new_fa", "lost_fa", "new_fa_same_winner",
               "new_fa_changed_winner", "delta_fpir", "delta_ci_low", "delta_ci_high"]])
adjacent = result["adjacent"]
display(adjacent.loc[adjacent.target_fpir == FOCUS_FPIR] if not adjacent.empty else adjacent)


## 4. 분포 상단과 최대 점수 변화의 분해

cosine에서 `최대 점수 변화 = 원본 승자에서의 점수 변화 + 새 후보 선택으로 얻은 점수 증가`입니다. 원본 승자가 그대로여도 오수락이 생길 수 있습니다. 빈 cohort의 평균은 0이 아니라 NaN입니다. 부동소수점 오차 허용치는 1e-6이며 작은 음의 선택 이득도 원값을 유지합니다.

분위수는 미등록 query별 최대값의 분포입니다. 원본/압축 ADC 점수는 서로 다른 척도임에 주의하세요. 이 분해는 관찰된 점수 변화의 항등식이지 양자화 기하학의 인과 규명이 아닙니다.


In [ ]:
tails = result["tails"]
display(tails.loc[(tails.target_fpir == FOCUS_FPIR) & (tails["quantile"] == 0.99)])
winners = result["winners"]
display(winners.loc[(winners.target_fpir == FOCUS_FPIR) & (winners.cohort == "new_fa")] if not winners.empty else winners)
display(focus[["compression_profile", "search_mode", "threshold_policy",
               "origin_threshold", "compressed_threshold",
               "score_effect_at_origin_threshold", "threshold_effect_after_score_change",
               "compressed_tpir20", "compressed_rank20"]])


## 5. 판정과 다음 단계

1. `frozen_origin`만 급증하고 재보정으로 줄어들면 점수·임계값 불일치와 일치하는 관측입니다. TPIR20/Rank20까지 회복했는지 별도로 확인하세요.
2. `new_fa_same_winner > 0`이면 후보 변경은 오수락 증가의 필요조건이 아닙니다.
3. 양쪽 복원에서만 큰 증가가 보여도 query 압축의 인과적 기여를 확정하지 않습니다. query-only 대조가 추가로 필요합니다.
4. 재보정 후에도 목표 초과가 남으면 calibration→test 전이 분석과 연결합니다. 이 노트북은 새로운 분할 안정성 실험을 수행하지 않습니다.

**후속 2차 미구현:** 정규화된 복원 오차 방향의 세 항 분해, cosine calibration 재생, query-only 대조. **후속 3차:** 여러 모델·데이터셋의 통합 보고. 모델 설정 변경은 지원하지만 모델 간 동일 cohort를 가정한 paired 비교는 하지 않습니다.

모든 rate/CI는 0~1 단위입니다. 퍼센트는 ×100, 차이의 ×100은 %p입니다. TPIR20의 새 cluster CI는 여기서 산출하지 않습니다.


In [ ]:
if WRITE_RESULTS:
    output_dir = write_diagnosis(result, OUTPUT_ROOT)
    print("New diagnostic artifacts:", output_dir)
else:
    print("Read-only: no diagnostic output written. Set WRITE_RESULTS=True to export.")
display({k: v for k, v in result["provenance"].items() if k != "inventory"})
